In [8]:
import sys
import os
import yaml
from langchain_core.documents import Document

# 1. 경로 설정 및 모듈 임포트
PROJECT_ROOT = "/home/taehoon/sprint-ai-mid-project_team3"
if os.path.abspath(PROJECT_ROOT) not in sys.path:
    sys.path.append(os.path.abspath(PROJECT_ROOT))

# [오류 해결 포인트] 클래스가 없어도 에러가 나지 않도록 함수(create_retriever)를 직접 불러옵니다.
from src.retriever_factory import create_retriever

# default.yaml 설정 파일을 직접 로드합니다.
CONFIG_PATH = os.path.join(PROJECT_ROOT, "config/default.yaml")
if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"설정 파일이 존재하지 않습니다: {CONFIG_PATH}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    full_config = yaml.safe_load(f)

retrieval_config = full_config.get("retrieval") or full_config.get("retriever", {})

# 2. 지정된 경로에서 sample_rfp.txt 파일을 읽어와 청킹 수행
FILE_PATH = os.path.join(PROJECT_ROOT, "samples/raw/sample_rfp.txt")
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"지정한 경로에 파일이 존재하지 않습니다: {FILE_PATH}")

with open(FILE_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

# [전처리] 텍스트를 문단(연속된 두 줄 줄바꿈) 단위로 분할하여 청크 생성
paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
processed_docs = []
chunks_for_factory = []

for idx, paragraph in enumerate(paragraphs):
    if idx % 2 == 0:
        org = "과학기술정보통신부"
        project = "지능형_AI_플랫폼_구축"
    else:
        org = "산업통상자원부"
        project = "에너지_빅데이터_분석"
    
    # 팩토리의 chunks 인자로 들어갈 기본 딕셔너리 포맷 생성
    chunks_for_factory.append({"text": paragraph, "org_name": org, "project_name": project})
        
    doc = Document(
        page_content=paragraph,
        metadata={
            "org_name": org,
            "project_name": project,
            "doc_id": f"DOC_RFP_{idx:03d}"
        }
    )
    processed_docs.append(doc)

print(f"총 {len(processed_docs)}개의 텍스트 문서 청크가 준비되었습니다.")


# 3. [오류 해결 포인트] 팩토리 함수(create_retriever)를 직접 호출하여 리트리버 생성
retriever = create_retriever(
    chunks=chunks_for_factory, 
    retrieval_config=retrieval_config, 
    profile="local"
)

# Chroma DB에 데이터 적재
retriever.add_documents(processed_docs)
print("Chroma DB에 sample_rfp 데이터 적재 완료!\n")


# 4. 메타데이터 필터링 통합 검색 검증
query_str = "사업의 추진 배경 및 기대 효과를 알려주세요"

print("=== 1. 필터 없이 유사도 전체 검색 (Top-3) ===")
results = retriever.retrieve(query=query_str, top_k=3)
for d in results:
    print(f"[{d.metadata['doc_id']} / {d.metadata['org_name']}] {d.page_content[:60]}...")

print("\n=== 2. 기관명 필터 적용 (과학기술정보통신부 데이터만 필터링) ===")
results_filtered = retriever.retrieve(query=query_str, org_name="과학기술정보통신부", top_k=2)
for d in results_filtered:
    print(f"[{d.metadata['org_name']} / {d.metadata['project_name']}] {d.page_content[:60]}...")

print("\n=== 3. 특정 문서 고유 ID 정밀 검색 (DOC_RFP_001 강제 지정) ===")
results_doc = retriever.retrieve(query=query_str, doc_id="DOC_RFP_001")
if results_doc:
    print(f"[{results_doc[0].metadata['doc_id']}] {results_doc[0].page_content}")


print("\n" + "="*50)
print("=== 5. (추가) Multi-Collection 방식을 적용한 하이브리드 컬렉션 검증 ===")
print("="*50)

# 1. 과기부 전용 리트리버 생성 (함수 직접 호출)
msit_retriever = create_retriever(
    chunks=chunks_for_factory, 
    retrieval_config=retrieval_config, 
    profile="local"
)

# 2. 산자부 전용 리트리버 생성 (함수 직접 호출)
motie_retriever = create_retriever(
    chunks=chunks_for_factory, 
    retrieval_config=retrieval_config, 
    profile="local"
)

# 3. 검색 시에는 해당 기관 리트리버에서 사업명(project_name) 필터만 걸고 검색
results_hybrid = msit_retriever.retrieve(query="보안 기술", project_name="AI보안사업")
print("Multi-Collection 하이브리드 테스트 호출 완료")

총 16개의 텍스트 문서 청크가 준비되었습니다.


/home/taehoon/sprint-ai-mid-project_team3/src/local_chroma_retriever.py:24: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

/home/taehoon/sprint-ai-mid-project_team3/src/local_chroma_retriever.py:34: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.vector_store = Chroma(


Chroma DB에 sample_rfp 데이터 적재 완료!

=== 1. 필터 없이 유사도 전체 검색 (Top-3) ===
[DOC_RFP_002 / 과학기술정보통신부] 1. 사업 개요...
[DOC_RFP_003 / 산업통상자원부] 사업명은 "2026년 공공 AI 학습지원 플랫폼 구축 사업"이며 발주기관은 가상디지털진흥원이다. 사업 목적은...
[DOC_RFP_000 / 과학기술정보통신부] [가상 제안요청서]...

=== 2. 기관명 필터 적용 (과학기술정보통신부 데이터만 필터링) ===
[과학기술정보통신부 / 지능형_AI_플랫폼_구축] 1. 사업 개요...
[과학기술정보통신부 / 지능형_AI_플랫폼_구축] [가상 제안요청서]...

=== 3. 특정 문서 고유 ID 정밀 검색 (DOC_RFP_001 강제 지정) ===
[DOC_RFP_001] 이 문서는 RAG 시스템 개발과 검색 성능 실험을 위해 작성한 가상 데이터입니다. 실제 기관, 사업 또는 입찰 공고와 관련이 없습니다.

=== 5. (추가) Multi-Collection 방식을 적용한 하이브리드 컬렉션 검증 ===


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Multi-Collection 하이브리드 테스트 호출 완료
